# Final Grand Baseline: Algebraic Hashing vs. State-of-the-Art
This notebook provides a clean, streamlined benchmark execution pipeline:
- Evaluates all 10 benchmark datasets (`cora`, `citeseer`, `pubmed`, `wikics`, `chameleon`, `squirrel`, `photo`, `computers`, `amazon-ratings`, `dblp`).
- Compares proposed algebraic hashing (`NyS-Binary` ) against state-of-the-art baselines (`node2binary`, `NodeSig`, `NodeSketch`, `Bi-GCN`, `Raw Features`).
- Evaluates across 9 downstream classifiers over 3 random seeds.


Alg_V1_Nystrom_Blend and Nys_Binary are same 

In [4]:
# ============================================================
# 0. IMPORTS & ENVIRONMENT SETUP
# ============================================================
import os, sys, time, warnings, importlib
warnings.filterwarnings('ignore')

# Force the TESTS/Github directory to be first in sys.path
BENCHMARK_DIR = r'./'
if BENCHMARK_DIR in sys.path:
    sys.path.remove(BENCHMARK_DIR)
sys.path.insert(0, BENCHMARK_DIR)

import torch
import numpy as np
import pandas as pd

for m in list(sys.modules.keys()):
    if m == 'torch_geometric' and not hasattr(sys.modules[m], 'typing'):
        del sys.modules[m]
try:
    import torch_geometric
    import torch_geometric.typing
except ImportError:
    pass

for mod in ['advanced_baselines', 'benchmark_new_algebraic']:
    if mod in sys.modules:
        del sys.modules[mod]

import advanced_baselines
import benchmark_new_algebraic

print(f'advanced_baselines loaded from: {advanced_baselines.__file__}')
print(f'Has fuse_alg_v1_randomized_blend: {hasattr(advanced_baselines, "fuse_alg_v1_randomized_blend")}')
print(f'Has fuse_alg_v1_random_blend: {hasattr(advanced_baselines, "fuse_alg_v1_random_blend")}')
print(f'benchmark_new_algebraic loaded from: {benchmark_new_algebraic.__file__}')

load_dataset = benchmark_new_algebraic.load_dataset
extract_features = benchmark_new_algebraic.extract_features
run_benchmark = benchmark_new_algebraic.run_benchmark
print_summary = benchmark_new_algebraic.print_summary
CLASSIFIER_REGISTRY = benchmark_new_algebraic.CLASSIFIER_REGISTRY
SEEDS = benchmark_new_algebraic.SEEDS
DEVICE = benchmark_new_algebraic.DEVICE

print(f'Device: {DEVICE}')
print(f'PyTorch version: {torch.__version__}')
print(f'Available classifiers ({len(CLASSIFIER_REGISTRY)}): {list(CLASSIFIER_REGISTRY.keys())}')


advanced_baselines loaded from: c:\Studies\MP2\TESTS\reviewing\Nys-Binary\advanced_baselines.py
Has fuse_alg_v1_randomized_blend: True
Has fuse_alg_v1_random_blend: True
benchmark_new_algebraic loaded from: c:\Studies\MP2\TESTS\reviewing\Nys-Binary\benchmark_new_algebraic.py
Device: cuda
PyTorch version: 2.5.1
Available classifiers (10): ['SVM', 'LogisticRegression', 'MLP', 'GCN', 'GraphSAGE', 'LINKX', 'SADP', 'SADP_Two_Way', 'SADP_Surrogate', 'H2GCN']


In [5]:
# ============================================================
# CONFIGURATION
# ============================================================

load_previous_checkpoint = False  # Set to True or False

load_cached_binary = False  # Set to True or False

# --- Load Direct CSV Option ---
# If True, loads results directly from `csv_to_load` instead of running run_benchmark.
# Even when the CSV contains all 10 datasets, the notebook will automatically
# filter down to only the datasets selected in `datasets` below!
load_from_csv = False  # Set to True to load precomputed CSV results
csv_to_load = os.path.join(BENCHMARK_DIR, "Results", "final_benchmark_results_latest.csv")

# --- Training Label Ratio ---
train_ratio = 0.70

# --- Datasets ---
datasets = [
    'cora',
    'citeseer',
    'pubmed',
    'wikics',
    'chameleon',
    'squirrel',
    'photo',
    'computers',
    'amazon-ratings',
    'dblp',


    ##### Highly Heterophilic Datasets #####
    # 'actor',
    # 'texas',
    # 'winconsin',
    # 'minesweeper',

    ### Large Datasets ###
    # 'ogbn-arxiv',
    # 'ogbn-products'
]

# --- Embedding Methods ---
methods = [
    #  External baselines 
    # 'node2binary',              # Official WWW 2025 Leiden random-walk binary embeddings 
    'NodeSig',                  # Official IEEE/ACM ASONAM 2022 binary random-walk signatures 
    # 'NodeSketch',               # Official ACM SIGKDD 2019 min-wise hashing binary embeddings (modulo 2 fix) 
    # 'Bi-GCN',                   # Official CVPR 2021 binary GCN trained 
    
    # # Internal baselines
    # 'Raw Features',
    # 'Raw Features Bin',
    # 'Top-K CE Adjacency',
    
    #  Proposed Algebraic Methods
    # 'Alg_V1_Nystrom_Blend',        # Official Proposed ALgorithm Nys-Binary
    # 'Nys_Binary_Unsupervised',
    # 'Nys_Binary_1pct',
    # 'Nys_Binary_10pct'

    # Other version of the proposed Methods
    # 'Alg_V1_SVD_Blend',         
    # 'Alg_V1_Randomized_Blend',  
    # 'Alg_V1_Chebyshev_Blend',  
]

# --- Downstream Classifiers (10 Models) ---
classifiers = {
    'LogisticRegression': CLASSIFIER_REGISTRY['LogisticRegression'],
    'MLP':          CLASSIFIER_REGISTRY['MLP'],
    'GCN':          CLASSIFIER_REGISTRY['GCN'],
    'GraphSAGE':    CLASSIFIER_REGISTRY['GraphSAGE'],
    'LINKX':        CLASSIFIER_REGISTRY['LINKX'],
    # 'SADP':         CLASSIFIER_REGISTRY['SADP'],
    # 'SADP_Two_Way': CLASSIFIER_REGISTRY['SADP_Two_Way'],
    # 'SADP_Surrogate': CLASSIFIER_REGISTRY['SADP_Surrogate'],
    # 'H2GCN':        CLASSIFIER_REGISTRY['H2GCN'],
}

# --- Random Seeds (3 seeds for statistical significance) ---
seeds = [42, 123, 77]

# --- Embedding Dimension ---
emb_dim = 256

# --- Device ---
device = DEVICE

# --- Print Configuration ---
print('=' * 70)
print('  BENCHMARK CONFIGURATION')
print('=' * 70)
print(f'Train Ratio:   {train_ratio:.3f} ({int(round(train_ratio*100))}% labels)')
print(f'Datasets ({len(datasets)}):    {datasets}')
print(f'Methods  ({len(methods)}):     {methods}')
print(f'Classifiers ({len(classifiers)}): {list(classifiers.keys())}')
print(f'Seeds    ({len(seeds)}):       {seeds}')
print(f'Emb Dim:       {emb_dim}')
print(f'Device:        {device}')
print(f'Load Checkpoint (✓/✗): {load_previous_checkpoint}')
print(f'Load Cached Binary: {load_cached_binary}')
print(f'Load Direct CSV: {load_from_csv} ({csv_to_load if load_from_csv else "N/A"})')
print('=' * 70)


  BENCHMARK CONFIGURATION
Train Ratio:   0.700 (70% labels)
Datasets (10):    ['cora', 'citeseer', 'pubmed', 'wikics', 'chameleon', 'squirrel', 'photo', 'computers', 'amazon-ratings', 'dblp']
Methods  (1):     ['NodeSig']
Classifiers (5): ['LogisticRegression', 'MLP', 'GCN', 'GraphSAGE', 'LINKX']
Seeds    (3):       [42, 123, 77]
Emb Dim:       256
Device:        cuda
Load Checkpoint (✓/✗): False
Load Cached Binary: False
Load Direct CSV: False (N/A)


In [6]:
# ============================================================
# RUN BENCHMARK OR LOAD EXISTING RESULTS
# ============================================================
import glob

if load_from_csv:
    if os.path.exists(csv_to_load):
        print(f"[Loading Engine] Loading results directly from: {csv_to_load}")
        df = pd.read_csv(csv_to_load)
    else:
        results_dir = os.path.join(BENCHMARK_DIR, "Results")
        csv_files = sorted(glob.glob(os.path.join(results_dir, "*.csv")), key=os.path.getmtime, reverse=True)
        if csv_files:
            print(f"[Loading Engine] Target file not found. Loading latest available: {csv_files[0]}")
            df = pd.read_csv(csv_files[0])
        else:
            raise FileNotFoundError(f"No result CSV found at {csv_to_load} or in {results_dir}")
else:
    print('Starting Full Benchmark...')
    print('=' * 70)
    df = run_benchmark(
        datasets=datasets,
        methods=methods,
        classifiers=classifiers,
        seeds=seeds,
        emb_dim=emb_dim,
        device_str=device,
        skip_existing=load_previous_checkpoint,
        train_ratio=train_ratio,
        load_cached_binary=load_cached_binary,
    )
    print('\n' + '=' * 70)
    print('  BENCHMARK COMPLETE!')
    print('=' * 70)

# ============================================================
# DYNAMIC FILTER: Keep ONLY the datasets selected in `datasets`
# Ensures that even when loading a full CSV containing other datasets,
# all downstream averages and reports are strictly computed over
# only the datasets you actually chose to run!
# ============================================================
if not df.empty:
    col_map = {
        'Dataset': 'dataset', 'Method': 'method', 'Classifier': 'classifier',
        'Accuracy': 'accuracy', 'Seed': 'seed', 'Gen_Time_s': 'gen_time',
        'gen_time_s': 'gen_time', 'GenTime': 'gen_time', 'F1_Macro': 'f1_macro',
        'F1_Micro': 'f1_micro'
    }
    df.rename(columns={k: v for k, v in col_map.items() if k in df.columns}, inplace=True)

    if 'dataset' in df.columns:
        df['dataset'] = df['dataset'].astype(str).str.lower().str.strip()
        active_datasets = [d.lower().strip() for d in datasets]
        df = df[df['dataset'].isin(active_datasets)].copy()

    if methods and 'method' in df.columns:
        active_methods = [m.strip() for m in methods]
        df = df[df['method'].isin(active_methods)].copy()

    eval_ds = sorted(df['dataset'].unique().tolist()) if 'dataset' in df.columns else []
    print('\n' + '=' * 70)
    print(f'  EVALUATED DATASET FILTER: {len(eval_ds)} datasets retained')
    print(f'  Datasets: {eval_ds}')
    print(f'  Methods:  {sorted(df["method"].unique().tolist()) if "method" in df.columns else []}')
    print(f'  Total Evaluation Rows: {len(df)}')
    print('=' * 70)


Starting Full Benchmark...
[Checkpoint Engine] Checkpoint loading DISABLED (skip_existing=False). Running fresh evaluations from scratch.

  Dataset: cora

  Seed: 42 | Device: cuda  (train=1895, test=813)
[NodeSig] Cache loading disabled (load_cache=False). Generating fresh binary embeddings for N=2708...
    NodeSig                 gen=   0.01s  shape=(2708, 256)
    Geometry                D_intra=124.2000 D_inter=127.4733 R=1.0264  
      [LogisticRegression  ]  Acc= 68.51%  F1m= 66.98%  train= 0.53s
      [MLP                 ]  Acc= 65.31%  F1m= 63.23%  train= 0.58s
      [GCN                 ]  Acc= 83.64%  F1m= 83.28%  train= 1.28s
      [GraphSAGE           ]  Acc= 83.03%  F1m= 82.53%  train= 1.08s
      [LINKX               ]  Acc= 70.23%  F1m= 68.38%  train= 2.03s

    [Checkpoint Saved] Seed 42 (5 records) saved to: c:\Studies\MP2\TESTS\reviewing\Nys-Binary\Results\results_cora_70pct_seed42_20261110_234227.csv
    [Checkpoint Saved] Cumulative run (5 records) saved to: c:\S

In [7]:
# ============================================================
# SAVE RESULTS TO CSV
# ============================================================
import os
import pandas as pd
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
results_dir = os.path.join(BENCHMARK_DIR, "Results")
os.makedirs(results_dir, exist_ok=True)

if not load_from_csv:
    # Save timestamped result file
    pct_tag = f"{int(round(train_ratio * 100))}pct"
    results_path = os.path.join(results_dir, f"final_benchmark_results_{pct_tag}_{timestamp}.csv")
    df.to_csv(results_path, index=False)
    print(f"Results saved to: {results_path}")

    # Also save latest consolidated results
    latest_path = os.path.join(results_dir, "final_benchmark_results_latest.csv")
    df.to_csv(latest_path, index=False)
    print(f"Latest results saved to: {latest_path}")
else:
    print(f"Loaded from CSV ({len(df)} rows across {len(df['dataset'].unique()) if 'dataset' in df.columns else 0} datasets). Checkpoint files preserved.")


Results saved to: ./Results\final_benchmark_results_70pct_20261110_234642.csv
Latest results saved to: ./Results\final_benchmark_results_latest.csv


In [8]:
# ============================================================
# DATASET-BY-DATASET SUMMARY REPORT
# ============================================================
import importlib
import benchmark_new_algebraic
import advanced_baselines
importlib.reload(advanced_baselines)
importlib.reload(benchmark_new_algebraic)
from benchmark_new_algebraic import print_summary

print_summary(df)



  DATASET: CORA (Mean ± Std % across seeds)
classifier           GCN     GraphSAGE         LINKX LogisticRegression           MLP
method                                                                               
NodeSig     82.98 ± 0.75  78.11 ± 6.42  69.86 ± 1.69       70.81 ± 1.99  67.73 ± 2.24

  DATASET: CITESEER (Mean ± Std % across seeds)
classifier           GCN     GraphSAGE         LINKX LogisticRegression           MLP
method                                                                               
NodeSig     61.49 ± 1.23  60.56 ± 0.87  45.38 ± 9.23       45.75 ± 0.79  49.98 ± 1.42

  DATASET: PUBMED (Mean ± Std % across seeds)
classifier           GCN     GraphSAGE         LINKX LogisticRegression           MLP
method                                                                               
NodeSig     73.07 ± 0.98  76.61 ± 0.63  74.80 ± 1.27       62.49 ± 0.14  63.93 ± 0.45

  DATASET: WIKICS (Mean ± Std % across seeds)
classifier           GCN     GraphSAGE

In [9]:
# ============================================================
# OVERALL AVERAGE ACCURACY & RUNTIME TABLES (MEAN ± STD)
# ============================================================
if 'method' in df.columns and 'classifier' in df.columns and 'accuracy' in df.columns:
    evaluated_datasets = sorted(df['dataset'].unique().tolist()) if 'dataset' in df.columns else []
    
    # 1. Dataset-Level Macro Average (First average across seeds per dataset, then mean ± std across datasets)
    ds_means = df.groupby(['dataset', 'method', 'classifier'])['accuracy'].mean().reset_index()
    macro_m = ds_means.pivot_table(values='accuracy', index='method', columns='classifier', aggfunc='mean')
    macro_s = ds_means.pivot_table(values='accuracy', index='method', columns='classifier', aggfunc='std', dropna=False).fillna(0.0)
    
    macro_table = pd.DataFrame(index=macro_m.index, columns=macro_m.columns)
    for col in macro_m.columns:
        macro_table[col] = [f"{mv:.2f} ± {sv:.2f}" for mv, sv in zip(macro_m[col], macro_s[col])]

    # 2. Pooled Average across all seeds and evaluated datasets
    pooled_m = df.pivot_table(values='accuracy', index='method', columns='classifier', aggfunc='mean')
    pooled_s = df.pivot_table(values='accuracy', index='method', columns='classifier', aggfunc='std', dropna=False).fillna(0.0)
    
    pooled_table = pd.DataFrame(index=pooled_m.index, columns=pooled_m.columns)
    for col in pooled_m.columns:
        pooled_table[col] = [f"{mv:.2f} ± {sv:.2f}" for mv, sv in zip(pooled_m[col], pooled_s[col])]

    print("=" * 80)
    print(f"  OVERALL AVERAGE ACCURACY across {len(evaluated_datasets)} EVALUATED DATASETS (Macro Mean ± Std)")
    print(f"  Datasets: {', '.join(evaluated_datasets)}")
    print("=" * 80)
    print(macro_table.to_string())
    print()

    print("=" * 80)
    print(f"  POOLED AVERAGE ACCURACY across all runs ({len(evaluated_datasets)} datasets × seeds)")
    print("=" * 80)
    print(pooled_table.to_string())
    print()

    if 'gen_time' in df.columns:
        print("=" * 80)
        print(f"  OVERALL EMBEDDING GENERATION TIME across {len(evaluated_datasets)} EVALUATED DATASETS")
        print("=" * 80)
        gen_df = df.groupby(['dataset', 'method', 'seed'])['gen_time'].first().reset_index()
        t_stats = gen_df.groupby('method')['gen_time'].agg(
            Mean_s='mean',
            Std_s='std',
            Median_s='median'
        ).fillna(0.0)
        time_table = pd.DataFrame(index=t_stats.index)
        time_table['Generation Time (Mean ± Std)'] = [
            f"{r['Mean_s']:>8.4f}s ± {r['Std_s']:>8.4f}s" for _, r in t_stats.iterrows()
        ]
        time_table['Median (s)'] = [f"{r['Median_s']:>8.4f}s" for _, r in t_stats.iterrows()]
        print(time_table.to_string())


  OVERALL AVERAGE ACCURACY across 10 EVALUATED DATASETS (Macro Mean ± Std)
  Datasets: amazon-ratings, chameleon, citeseer, computers, cora, dblp, photo, pubmed, squirrel, wikics
classifier            GCN      GraphSAGE          LINKX LogisticRegression            MLP
method                                                                                   
NodeSig     70.39 ± 18.08  71.02 ± 17.59  71.18 ± 14.19      64.09 ± 17.73  65.45 ± 15.55

  POOLED AVERAGE ACCURACY across all runs (10 datasets × seeds)
classifier            GCN      GraphSAGE          LINKX LogisticRegression            MLP
method                                                                                   
NodeSig     70.39 ± 17.46  71.02 ± 17.08  71.18 ± 15.33      64.09 ± 17.12  65.45 ± 15.04

  OVERALL EMBEDDING GENERATION TIME across 10 EVALUATED DATASETS
        Generation Time (Mean ± Std) Median (s)
method                                         
NodeSig          0.0162s ±   0.0118s    0.0135s


In [10]:
# ============================================================
# MEAN GENERATION TIME ACROSS DATASETS (MEAN ± STD)
# ============================================================
work_df = df.copy()
col_map = {
    'Dataset': 'dataset',
    'Method': 'method',
    'Seed': 'seed',
    'Gen_Time_s': 'gen_time',
    'gen_time_s': 'gen_time',
    'GenTime': 'gen_time',
}
work_df.rename(columns={k: v for k, v in col_map.items() if k in work_df.columns}, inplace=True)

if 'gen_time' in work_df.columns and 'dataset' in work_df.columns and 'method' in work_df.columns:
    gen_df = work_df.groupby(['dataset', 'method', 'seed'])['gen_time'].first().reset_index()
    evaluated_datasets = sorted(gen_df['dataset'].unique().tolist())

    # ---  Mean Time Across Evaluated Datasets (Mean ± Std for evaluated datasets) ---
    ds_means = gen_df.groupby(['method', 'dataset'])['gen_time'].mean().reset_index()
    across_ds = ds_means.groupby('method')['gen_time'].agg(
        Mean_s='mean',
        Std_s='std',
        Median_s='median',
        Min_s='min',
        Max_s='max'
    ).fillna(0.0).sort_values(by='Mean_s', ascending=True)

    summary_table = pd.DataFrame(index=across_ds.index)
    summary_table['Generation Time (Mean ± Std)'] = [
        f"{r['Mean_s']:>8.4f}s ± {r['Std_s']:>8.4f}s" for _, r in across_ds.iterrows()
    ]
    summary_table['Median (s)'] = [f"{r['Median_s']:>8.4f}s" for _, r in across_ds.iterrows()]
    summary_table['Min (s)'] = [f"{r['Min_s']:>8.4f}s" for _, r in across_ds.iterrows()]
    summary_table['Max (s)'] = [f"{r['Max_s']:>8.4f}s" for _, r in across_ds.iterrows()]

    print('=' * 80)
    print(f'  MEAN EMBEDDING GENERATION TIME ACROSS {len(evaluated_datasets)} EVALUATED DATASETS')
    print(f'  Datasets: {evaluated_datasets}')
    print('=' * 80)
    print(summary_table.to_string())
    print()

    # --- Dataset-by-Dataset Breakdown (Mean ± Std across seeds per dataset) ---
    p_mean = gen_df.pivot_table(values='gen_time', index='method', columns='dataset', aggfunc='mean').reindex(across_ds.index)
    p_std = gen_df.pivot_table(values='gen_time', index='method', columns='dataset', aggfunc='std').fillna(0.0).reindex(across_ds.index)

    time_per_dataset = pd.DataFrame(index=p_mean.index, columns=p_mean.columns)
    for col in p_mean.columns:
        time_per_dataset[col] = [f"{m:.4f}s ± {s:.4f}s" for m, s in zip(p_mean[col], p_std[col])]

    print('=' * 80)
    print('  DATASET-BY-DATASET GENERATION TIME BREAKDOWN (Mean ± Std seconds across seeds)')
    print('=' * 80)
    print(time_per_dataset.to_string())


  MEAN EMBEDDING GENERATION TIME ACROSS 10 EVALUATED DATASETS
  Datasets: ['amazon-ratings', 'chameleon', 'citeseer', 'computers', 'cora', 'dblp', 'photo', 'pubmed', 'squirrel', 'wikics']
        Generation Time (Mean ± Std) Median (s)    Min (s)    Max (s)
method                                                               
NodeSig          0.0162s ±   0.0101s    0.0140s    0.0064s    0.0414s

  DATASET-BY-DATASET GENERATION TIME BREAKDOWN (Mean ± Std seconds across seeds)
dataset     amazon-ratings          chameleon           citeseer          computers               cora               dblp              photo             pubmed           squirrel             wikics
method                                                                                                                                                                                               
NodeSig  0.0414s ± 0.0190s  0.0077s ± 0.0042s  0.0103s ± 0.0057s  0.0193s ± 0.0055s  0.0064s ± 0.0046s  0.0188s ± 0.0092s  0